In [2]:
import torch
import torch.nn.functional as F
from mkunet_network import MK_UNet

In [3]:
model = MK_UNet()
x = torch.randn(1, 3, 352, 352)
print(x.shape)

torch.Size([1, 3, 352, 352])


# The Encoder

In [4]:
# Step 1: the MKIR block (the kernels for this stage)
out = model.encoder1(x)
print("after encoder1: ", out.shape)

# Step 2: 2x2 max pooling, stride-2
out = F.max_pool2d(out, 2, 2)
print("after pooling:", out.shape)

# Save the skip copy
t1 = out

after encoder1:  torch.Size([1, 16, 352, 352])
after pooling: torch.Size([1, 16, 176, 176])


In [5]:
# Stage 2 encoding
out = F.max_pool2d(model.encoder2(out), 2, 2)
print("after Stage2:", out.shape) # Expected output: torch.Size([1, 32, 88, 88])
t2 = out

# Stage 3 encoding
out = F.max_pool2d(model.encoder3(out), 2, 2)
print("after Stage3:", out.shape) # Expected output: torch.Size([1, 64, 44, 44])
t3 = out

# Stage 4 encoding
out = F.max_pool2d(model.encoder4(out), 2, 2)
print("after Stage4:", out.shape) # Expected output: torch.Size([1, 96, 22, 22])
t4 = out

# Bottleneck (no skip copy is saved)
out = F.max_pool2d(model.encoder5(out), 2, 2)
print("after Bottleneck:", out.shape) # Expected output: torch.Size([1, 160, 11, 11])

after Stage2: torch.Size([1, 32, 88, 88])
after Stage3: torch.Size([1, 64, 44, 44])
after Stage4: torch.Size([1, 96, 22, 22])
after Bottleneck: torch.Size([1, 160, 11, 11])


# The Decoder

In [6]:
# Attention steps
out = model.CA1(out) * out
out = model.SA(out) * out
print("after attention:", out.shape) # Expected output: torch.Size([1, 160, 11, 11])

# Decoder1: an MKIR block that changes the page count
out = model.decoder1(out)
print("after decoder1:", out.shape) # Expected output: torch.Size([1, 96, 11, 11])

# Unsampling: doubling the size, the ReLU
out = F.relu(F.interpolate(out, scale_factor=(2,2), mode="bilinear"))
print("after upsample:", out.shape) # Expected output: torch.Size([1, 96, 22, 22])

# The gate on the skip copy, keyed by the decoder signal
t4 = model.AG1(g=out, x=t4)
print("gated t4:     ", t4.shape) # Expected output: torch.Size([1, 96, 22, 22])

# Add the gated skip copy in
out = torch.add(out, t4)
print("after add:    ", out.shape) # Expected output: torch.Size([1, 96, 22, 22])

after attention: torch.Size([1, 160, 11, 11])
after decoder1: torch.Size([1, 96, 11, 11])
after upsample: torch.Size([1, 96, 22, 22])
gated t4:      torch.Size([1, 96, 22, 22])
after add:     torch.Size([1, 96, 22, 22])


In [7]:
import torch.nn as nn
from mkunet_network import GroupedAttentionGate

# 1. Shrink the raw t3
diag = F.max_pool2d(t3, 2, 2)
print("after pool:", diag.shape)    # guess: torch.Size([1,64, 22, 22])

# 2. 1x1 kernel layer: 64 pages in, 96 pages out
convert = nn.Conv2d(64, 96, kernel_size=1, bias=False)
diag = convert(diag)
print("after 1x1: ", diag.shape)    # guess: torch.Size([1,96, 22, 22])

# 3. A new gate, built with the same settings as AG1
gate = GroupedAttentionGate(F_g=96, F_l=96, F_int=48, kernel_size=3, groups=48)
diag = gate(g=out, x=diag)
print("after gate:", diag.shape)    # guess: torch.Size([1, 96, 22, 22])

# 4. Add it to the decoder signal
combined = torch.add(out, diag)
print("after add: ", combined.shape)  # guess: torch.Size([1, 96, 22, 22])

after pool: torch.Size([1, 64, 22, 22])
after 1x1:  torch.Size([1, 96, 22, 22])
after gate: torch.Size([1, 96, 22, 22])
after add:  torch.Size([1, 96, 22, 22])


In [8]:
# Decoder stage 2, starting from the combined 22x22 result
out = combined
out = model.CA2(out) * out
out = model.SA(out) * out
out = F.relu(F.interpolate(model.decoder2(out), scale_factor=(2, 2), mode="bilinear"))
print("decoder signal:", out.shape)

# Standard skip copy, through its own gate
t3_gated = model.AG2(g=out, x=t3)

# The second diagonal line: t2 into this stage
diag2 = F.max_pool2d(t2, 2, 2)
convert2 = nn.Conv2d(32, 64, kernel_size=1, bias=False)
diag2 = convert2(diag2)
gate2 = GroupedAttentionGate(F_g=64, F_l=64, F_int=32, kernel_size=3, groups=32)
diag2 = gate2(g=out, x=diag2)
print("diagonal 2:    ", diag2.shape)

# Add both copies in
out = out + t3_gated + diag2
print("after add:     ", out.shape)

# Decoder stage 3, starting from the combined 44x44 result
out = model.CA3(out) * out
out = model.SA(out) * out
out = F.relu(F.interpolate(model.decoder3(out), scale_factor=(2,2), mode="bilinear"))
print("decoder signal:", out.shape)

t2_gated = model.AG3(g=out, x=t2)

diag3 = F.max_pool2d(t1, 2, 2)
convert3 = nn.Conv2d(16, 32, kernel_size=1, bias=False)
diag3 = convert3(diag3)
gate3 = GroupedAttentionGate(F_g=32, F_l=32, F_int=16, kernel_size=3, groups=16)
diag3 = gate3(g=out, x=diag3)
print("diagonal 3:    ", diag3.shape)

out = out + t2_gated + diag3
print("after add:     ", out.shape)

out = model.CA4(out) * out
out = model.SA(out) * out
print("after attention:", out.shape) # Expected output: torch.Size([1, 32, 88, 88])

# Unsampling: doubling the size, the ReLU
out = F.relu(F.interpolate(model.decoder4(out), scale_factor=(2,2), mode="bilinear"))
print("after upsample:", out.shape) # Expected output: torch.Size([1, 16, 176, 176])

# Add the gated skip copy in
t1 = model.AG4(g=out, x=t1)
out = torch.add(out, t1)
print("after add:    ", out.shape) # Expected output: torch.Size([1, 16, 176, 176])

decoder signal: torch.Size([1, 64, 44, 44])
diagonal 2:     torch.Size([1, 64, 44, 44])
after add:      torch.Size([1, 64, 44, 44])
decoder signal: torch.Size([1, 32, 88, 88])
diagonal 3:     torch.Size([1, 32, 88, 88])
after add:      torch.Size([1, 32, 88, 88])
after attention: torch.Size([1, 32, 88, 88])
after upsample: torch.Size([1, 16, 176, 176])
after add:     torch.Size([1, 16, 176, 176])


In [9]:
out = model.CA5(out) * out
out = model.SA(out) * out
out = F.relu(F.interpolate(model.decoder5(out), scale_factor=(2,2), mode="bilinear"))
print("after decoder5:", out.shape) # Expected output: torch.Size([1,16,352, 352])

p4 = model.out4(out)
print("mask:          ", p4.shape) # Expected output: torch.Size([1, 1, 352, 352])

after decoder5: torch.Size([1, 16, 352, 352])
mask:           torch.Size([1, 1, 352, 352])


# Create a reference from the original model

In [10]:
from mkunet_original import MK_UNet as OriginalMKUNet

torch.manual_seed(0)
model = OriginalMKUNet()
model.eval()
x = torch.randn(1, 3, 352, 352)
with torch.no_grad():
    baseline = model(x)[0]
torch.save(baseline, "baseline_output.pt")

# Check the new model's baseline matches the original models baseline (the reference of the original model)

In [12]:
import torch
from mkunet_network import MK_UNet

def count(m):
    # total number of faders in the model
    return sum(p.numel() for p in m.parameters())

# Check 1: switch off matches the baseline recording
torch.manual_seed(0)
model_off = MK_UNet(diagonal_skips=False)
model_off.eval()
x = torch.randn(1, 3, 352, 352)
with torch.no_grad():
    off_mask = model_off(x)[0]
base = torch.load("baseline_output.pt")
print("off: faders", count(model_off), "| matches baseline:", torch.equal(off_mask, base))
print("biggest difference:", (off_mask - base).abs().max().item())

# Check 2 and 3: switch on
model_on = MK_UNet(diagonal_skips=True)
print("on:  faders", count(model_on))
print("color mask:    ", model_on(torch.randn(1, 3, 352, 352))[0].shape)
print("grayscale mask:", model_on(torch.randn(1, 1, 352, 352))[0].shape)

off: faders 315566 | matches baseline: True
biggest difference: 0.0
on:  faders 328407
color mask:     torch.Size([1, 1, 352, 352])
grayscale mask: torch.Size([1, 1, 352, 352])
